# [1교시]

In [1]:
# import shutil

# # 기존에 불완전하게 생성된 폴더 삭제
# shutil.rmtree('./local_qwen_model', ignore_errors=True)
# print("기존 폴더 삭제 완료. 아래 셀을 다시 실행해 모델을 다운로드하세요.")

In [2]:
%pip install tiktoken sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
import torch
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

In [4]:
# 벡터데이터베이스 로드
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_db = Chroma(persist_directory="./chroma_db_session", embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
local_model_dir = './local_qwen_model'

In [6]:
if not os.path.exists(local_model_dir):
    os.makedirs(local_model_dir, exist_ok=True)
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype = torch.float32)
    _tokenizer.save_pretrained(local_model_dir)
    _model.save_pretrained(local_model_dir)

tokenizer = AutoTokenizer.from_pretrained(local_model_dir)
model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype = torch.float32,
                                             device_map='cpu',low_cpu_mem_usage=True)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [7]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    max_length=None,
    temperature=0.1,
    do_sample=True,
    clean_up_tokenization_spaces=False,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'max_length', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
class GraphState(TypedDict):
    question:str
    context:str
    generation:str
    source_info:str

def retrieve(state:GraphState):
    print('--retrieve node --')
    question = state['question']
    docs = retriever.invoke(question)
    contexts,sources = [],[]
    for i, doc in enumerate(docs):
        # 텍스트데이터 포멧팅
        contexts.append(f'[문서 {i+1}] {doc.page_content}')
        # 메타데이터 추출
        filename = doc.metadata.get('filename', 'N/A')
        f_type = doc.metadata.get('file_type', 'N/A')
        cat = doc.metadata.get('category', 'N/A')
        sources.append(f"-문서 {i+1} [파일명:{filename} / 유형 : {f_type} / 카테고리 : {cat} ]")
    contexts_str = '\n\n'.join(contexts)
    sources_str = '\n\n'.join(sources)
    print(f'Retrieve {len(docs)} documents')
    return {"context":contexts_str, "source_info":sources_str}

def generate(state:GraphState):
    print('--generate node --')
    question = state['question']
    context = state['context']
    # LLM을 위한 프롬프트 템플릿
    messages = [
        {'role':'system', 'content':"주어진 문장을 참고해서 사용자의 질문에 한국어로 정확하고 간결하게 답변하세요"},
        {'role':'user','content':f'[본문]\n{context}\n\n[질문]\n{question}'}
    ]
    prompt = tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)
    response = llm.invoke(prompt)
    return {'generation':response.strip(), "source_info":state["source_info"]}

# [2교시]

In [9]:
workflow = StateGraph(GraphState)
workflow.add_node('retrieve', retrieve)
workflow.add_node('generate', generate)
workflow.set_entry_point('retrieve')
workflow.add_edge('retrieve', 'generate')
workflow.add_edge('generate', END)
app = workflow.compile()

In [10]:
# user_question = input("질문\n")
# inputs = {'question':user_question}
# final_state = app.invoke(inputs)
# print('\n========= 답변 =======')
# print(final_state['generation'])
# print('\n========= 출처정보 =======')
# print(final_state['source_info'])
# print('\n=========================')

In [ ]:
final_state.keys()

In [ ]:
print(final_state['source_info'])

# [3교시]

In [ ]:
# !pip install rouge kiwipiepy lm-eval

In [ ]:
# 정량적 평가 BLEU(기계번역 성능평가) ROUGE
# BLEU 예측 문장과 정답 문장이 얼마나 많은 n-gram을 공유하는지.
# BLEU-1 : unigram
# 단점 : 동의어를 인식 못함 The car is fast / The automobile is quick

# ROUGE(문서요약 평가)
# n-gram을 비교... Recall 중심  The cat sat on the mat / The cat sat
# 정답 : 6개, 예측 : 3개    6 / 3 = 0.5 --> recall  3 / 3 --> 1.0 Precision 
# ROUGE-1 : unigram... 

# 최근에는 다양한 평가지표가 사용 BERTScore... LLM-as-a-judge

In [12]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

Kiwi = Kiwi() # 한국어 토큰화
# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in Kiwi.tokenize(text)]

# 스무딩 기법 점수가 불필요하게 0이되는 문제를 완화하기위해 사용
def evaluate_ngram_improved(reference, candidate):
    smothie = SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    bleu_score = sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smothie)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str, ref_str)[0]
    return bleu_score, scores

ref = '파리를 여행할 때는 에펠타보가 루브르 박물관을 꼭 방문해야 합니다.'
cand = '파리 여행 시 에펠탑과 루브르 박문관은 반드시 가봐야 할 명소입니다.'

bleu, rouge_rs = evaluate_ngram_improved(ref, cand)

print(f'BLEU : {bleu}')
print(f"ROUGE-1 F1: {rouge_rs['rouge-1']['f']:.4f}")

BLEU : 0.05681443412487883
ROUGE-1 F1: 0.4571


In [24]:
from lm_eval.tasks import TaskManager
task_manager = TaskManager()
all_tasks = list(task_manager.task_index.keys())
all_tasks

['aclue',
 'aclue_ancient_chinese_culture',
 'aclue_ancient_literature',
 'aclue_ancient_medical',
 'aclue_ancient_phonetics',
 'aclue_basic_ancient_chinese',
 'aclue_couplet_prediction',
 'aclue_homographic_character_resolution',
 'aclue_named_entity_recognition',
 'aclue_poetry_appreciate',
 'aclue_poetry_context_prediction',
 'aclue_poetry_quality_assessment',
 'aclue_poetry_sentiment_analysis',
 'aclue_polysemy_resolution',
 'aclue_reading_comprehension',
 'aclue_sentence_segmentation',
 'acp_areach_bool',
 'acp_bool_cot_2shot',
 'acp_bench',
 'acp_app_bool',
 'acp_just_bool',
 'acp_land_bool',
 'acp_prog_bool',
 'acp_reach_bool',
 'acp_val_bool',
 'acp_areach_gen',
 'acp_gen_2shot',
 'acp_bench_hard',
 'acp_app_gen',
 'acp_just_gen',
 'acp_land_gen',
 'acp_nexta_gen',
 'acp_prog_gen',
 'acp_reach_gen',
 'acp_val_gen',
 'acp_areach_gen_with_pddl',
 'acp_gen_2shot_with_pddl',
 'acp_bench_hard_with_pddl',
 'acp_app_gen_with_pddl',
 'acp_just_gen_with_pddl',
 'acp_land_gen_with_pddl',

In [25]:
[task for task in all_tasks if 'hellaswag' in task]

['arabic_leaderboard_arabic_mt_hellaswag',
 'arabic_mt_hellaswag',
 'arabic_leaderboard_arabic_mt_hellaswag_light',
 'arabic_mt_hellaswag_light',
 'darijahellaswag',
 'egyhellaswag',
 'french_bench_hellaswag',
 'hellaswag',
 'kobest_hellaswag',
 'metabench_hellaswag',
 'metabench_hellaswag_subset',
 'metabench_hellaswag_permute',
 'metabench_hellaswag_secondary',
 'metabench_hellaswag_secondary_permute',
 'hellaswag_ar',
 'hellaswag_multilingual',
 'hellaswag_bn',
 'hellaswag_ca',
 'hellaswag_da',
 'hellaswag_de',
 'hellaswag_es',
 'hellaswag_eu',
 'hellaswag_fr',
 'hellaswag_gu',
 'hellaswag_hi',
 'hellaswag_hr',
 'hellaswag_hu',
 'hellaswag_hy',
 'hellaswag_id',
 'hellaswag_it',
 'hellaswag_kn',
 'hellaswag_ml',
 'hellaswag_mr',
 'hellaswag_ne',
 'hellaswag_nl',
 'hellaswag_pt',
 'hellaswag_ro',
 'hellaswag_ru',
 'hellaswag_sk',
 'hellaswag_sr',
 'hellaswag_sv',
 'hellaswag_ta',
 'hellaswag_te',
 'hellaswag_uk',
 'hellaswag_vi']

In [15]:
# LM-Evaluation_harness 벤치 마크 평가
# 모델의 추론 능력을 벤치마크 데이터셋(hellaswag)을 통해 accuracy를 측정
import lm_eval
try:
    result = lm_eval.simple_evaluate(
        model="hf",
        model_args = "pretrained=skt/kogpt2-base-v2,dtype=float32",
        tasks=['hellaswag'],
        device="cpu",
        limit=2
        )
    print("LM-Eval 정확도(acc): {result['results']['hallaswag']['acc, none']}")
except Exception as e:
    print(e)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/24.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/6.11M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/10042 [00:00<?, ? examples/s]

Map:   0%|          | 0/39905 [00:00<?, ? examples/s]

Map:   0%|          | 0/10042 [00:00<?, ? examples/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:01<00:00,  4.88it/s]


LM-Eval 정확도(acc): {result['results']['hallaswag']['acc, none']}


In [26]:
# 정답형태가 고정되어 있지않은 생성형 태스크의 경우 상용모델(gpt)을 판단기준으로 활용
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)
client = OpenAI()
def evaluate_with_gpt(prompt, generated_text):
    eval_prompt = f'''
다음 질문과 답변을 보고 정확성, 유창성, 관련성을 기준으로 1~5점 사이의 점수와 이류를 작성해주세요
질문:{prompt}
답변:{generated_text}
'''
    try:
        response = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages=[{'role':'user','content':eval_prompt}],
            max_completion_tokens = 150
        )
        return response.choices[0].message.content
    except Exception as e:
        return e
print(evaluate_with_gpt('프랑시의 명소는?','파리 여행시 에팔탑과 루브르 박물관은 반드시 가봐야 할 명소입니다.')    )

**점수: 2/5**

- **정확성(2점):** 질문은 “프랑시의 명소”인데, 답변은 *파리*의 에팔탑과 루브르를 언급했습니다. 프랑스의 명소로는 맞을 수 있지만, “프랑시(프랑스) 전반”에 대한 답변이라기보다 특정 지역(파리)에 한정되어 정확성은 부분적입니다.
- **유창성(3점):** 문장이 자연스럽고 의미 전달이 됩니다. 다만 “에팔탑”은 일반적으로 많이 쓰는 표기(예: 에펠탑)와 달


# [4교시]

In [39]:
# BASE모델 , 파인튜닝 Instruct 성능 비교
# Qween2.5  Qween2.5_instruct
from transformers import AutoModelForCausalLM, AutoTokenizer

prompt = '프랑스 파리를 여행하려고 해 꼭 가봐야 할 명소를 두 곳만 추천해줘'
base_model_id = 'Qwen/Qwen2.5-0.5B'
instruct_model_id = 'Qwen/Qwen2.5-0.5B-Instruct'

def generate(model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)    
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        pad_token_id = tokenizer.eos_token_id,
        temperature = 0.1,
        do_sample=True,
        max_new_tokens=40
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [40]:
print(f'질문 : {prompt}')
base_output = generate(base_model_id)
print(f'[Base모델 출력 : {base_output}]')
instruct_output = generate(instruct_model_id)
print(f'[Instruct 모델 출력 : {instruct_output}]')

질문 : 프랑스 파리를 여행하려고 해 꼭 가봐야 할 명소를 두 곳만 추천해줘


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Base모델 출력 : 프랑스 파리 여행을 위해 여행을 꼭 가봐야 할 명소를 추천해드리겠습니다. 1. 루이스 파리 (Rouen]


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Instruct 모델 출력 : 프랑스의 파리에서 가장 인기 있는 명소 중 하나로는 "파리의 중심지인 로마"가 있습니다. 이곳은 프랑스의 수도이며,]


In [44]:
# 정량적 평가 BL, ROUGE
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

Kiwi = Kiwi()  # 한국어 토큰화
# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in Kiwi.tokenize(text)]

# 스무딩기법 점수가 불필요하게 0이되는 문제를 완화하기위해 사용
def evaluate_ngram(reference, candidate):
    smothie =  SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    bleu_score = sentence_bleu(ref_tokens,cand_tokens,smoothing_function=smothie)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str,ref_str)[0]
    rouge_f1 = scores['rouge-1']['f']
    return scores,rouge_f1

ref = '파리의 명소로는 에펠탑과 루브르 박물관을 추천합니다.'
b_bleu, b_rouge = evaluate_ngram(ref,base_output)
i_bleu, i_rouge = evaluate_ngram(ref,instruct_output)

print(f'Base모델 BLEU : {b_bleu} ROUGE-1 : {b_rouge}')
print(f'Instruct모델 BLEU : {i_bleu} ROUGE-1 : {i_rouge}')

Base모델 BLEU : {'rouge-1': {'r': 0.38461538461538464, 'p': 0.22727272727272727, 'f': 0.285714281044898}, 'rouge-2': {'r': 0.08333333333333333, 'p': 0.038461538461538464, 'f': 0.05263157462603914}, 'rouge-l': {'r': 0.3076923076923077, 'p': 0.18181818181818182, 'f': 0.22857142390204094}} ROUGE-1 : 0.285714281044898
Instruct모델 BLEU : {'rouge-1': {'r': 0.38461538461538464, 'p': 0.20833333333333334, 'f': 0.27027026571219875}, 'rouge-2': {'r': 0.16666666666666666, 'p': 0.06666666666666667, 'f': 0.09523809115646274}, 'rouge-l': {'r': 0.3076923076923077, 'p': 0.16666666666666666, 'f': 0.21621621165814472}} ROUGE-1 : 0.27027026571219875


In [45]:
# LM-Evaluation-Harness평가
# 모델의 추론 능력을 벤치마크 데이터셋(hellaswag)을 통해 accuracy를 측정
import lm_eval
def lm_eval_score(model_id):
    try:
        result = lm_eval.simple_evaluate(
            model="hf",
            model_args = f"pretrained={model_id},dtype=float32",
            tasks=['hellaswag'],
            device="cpu",
            limit=2
        )
        print(f"LM-Eval 정확도(acc): {result['results']['hellaswag']['acc,none']}")
    except Exception as e:
        print(e)
lm_eval_score(base_model_id)
print('='*100)
lm_eval_score(instruct_model_id)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:03<00:00,  2.63it/s]
pretrained=pretrained=Qwen/Qwen2.5-0.5B-Instruct,dtype=float32 appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).


LM-Eval 정확도(acc): 0.0


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:03<00:00,  2.47it/s]


LM-Eval 정확도(acc): 0.0


# [5교시]

In [46]:
# GPT를 활용한 정성평가(LLM-as-a-Judge)
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)
client = OpenAI()

In [51]:
def evaluate_with_gpt(generate_text):
    eval_prompt = f'''
질문:{prompt}
답변:{generate_text}
위 질문에 대한 답변의 정확성, 유창성을 1~5점으로 평가하고 이유를 100자 이내로 작성하세요.
'''
    try:
        response = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages = [{'role':'user', 'content':eval_prompt}],
            max_completion_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return e
print('BASE model gpt eval')
print(evaluate_with_gpt(base_output))
print('\nInstruct model gpt eval')
print(evaluate_with_gpt(instruct_output))

BASE model gpt eval
평가: **1/5**  
이유: **명소 2곳이 아니고(루앙 1곳만), 파리 명소가 부정확하며 문장이 중간에 끊겼습니다.**

Instruct model gpt eval
평가: 1/5  
이유(100자 이내): 로마가 파리 중심지라는 오류. 명소도 2곳이 아니라 1곳만 제시. 유창성도 낮음.


In [ ]:
python -m pip install --upgrade pip 